# Семинар 9. DDL, DML и представления

Банк ведёт журнал кредитных заявок. Вы создадите таблицы, загрузите пробные данные,
исправите статусы и дубликаты, а затем подготовите два отчёта.

**Оценка.** Семь задач на **10 баллов**. Баллы указаны в заголовках.

**Как работать.** Запустите учебное окружение из `README.md` репозитория курса.
Выполните ячейку подключения один раз и затем двигайтесь по порядку: задачи
используют результаты предыдущих шагов. Начинайте SQL-ячейку с `%%sql`.
Для каждого пункта есть отдельная ячейка. В пункте с серией изменений схемы
можно записать несколько команд подряд через точку с запятой.

**Учебная схема.** Первая ячейка создаёт отдельную схему для текущего ядра.
В задачах пишите имена таблиц без префикса: соединение уже настроено на эту схему.
Чтобы начать с чистого листа, выполните необязательную ячейку очистки в конце,
верните в ней `CLEAN_UP = False` и заново запустите ноутбук сверху.
Все имена и суммы придуманы для практикума.

In [ ]:
import sys
from pathlib import Path
from uuid import uuid4

magic_directory = Path("../jupyter")
if not (magic_directory / "sql_magic.py").is_file():
    magic_directory = Path("../../common")
sys.path.insert(0, str(magic_directory.resolve()))
%load_ext sql_magic

if "practice_schema" not in globals():
    practice_schema = f"seminar_9_{uuid4().hex}"

get_ipython().run_cell_magic("sql", "", f"""
CREATE SCHEMA IF NOT EXISTS {practice_schema};
SET search_path TO {practice_schema}, pg_catalog;
SELECT current_schema() AS practice_schema;
""")

## Памятка

| Что сделать | Конструкция |
|---|---|
| Создать таблицу | `CREATE TABLE имя (столбец тип ограничение, ...)` |
| Получить следующий идентификатор | `GENERATED ALWAYS AS IDENTITY` |
| Задать ссылку и поведение при изменениях | `REFERENCES таблица (ключ) ON UPDATE CASCADE ON DELETE RESTRICT` |
| Изменить столбец | `ALTER TABLE … ADD COLUMN`, `RENAME COLUMN`, `ALTER COLUMN … TYPE`, `DROP COLUMN` |
| Задать значение по умолчанию или обязательность | `ALTER TABLE … ALTER COLUMN … SET DEFAULT значение`, `… SET NOT NULL` |
| Добавить или снять ограничение | `ALTER TABLE … ADD CONSTRAINT имя CHECK (условие)`, `… DROP CONSTRAINT имя` |
| Очистить таблицу | `TRUNCATE TABLE имя` |
| Удалить объект | `DROP TABLE IF EXISTS имя`, `DROP VIEW имя` |
| Вставить или изменить строки | `INSERT INTO … VALUES …`, `INSERT INTO … SELECT …`, `UPDATE … SET … WHERE …` |
| Удалить строки | `DELETE FROM … WHERE …` |
| Создать или заменить запрос | `CREATE VIEW имя AS SELECT …`, `CREATE OR REPLACE VIEW имя AS SELECT …` |
| Сохранить или обновить результат | `CREATE MATERIALIZED VIEW имя AS SELECT …`, `REFRESH MATERIALIZED VIEW имя` |

- `DEFAULT` применяется, когда столбец пропущен или указано слово `DEFAULT`.
  Для обязательного значения добавляйте `NOT NULL`.
- В `CREATE OR REPLACE VIEW` сохраняйте имена, порядок и типы существующих столбцов.
- `EXPLAIN (ANALYZE, COSTS OFF)` перед `SELECT` выполнит запрос и покажет план.
  На маленьком наборе данных сравнивайте операции плана, поскольку время меняется между запусками.
- `generate_series(начало, конец, INTERVAL '1 week')` строит календарь.
  Приведите значения к `date`. `date_trunc('week', дата)::date` даёт понедельник.
- `LAG(значение, 4) OVER (ORDER BY week_start)` берёт значение на четыре строки раньше.
  Чтобы это означало четыре недели, в ряду нужна строка для каждой недели.
  `NULLIF(база, 0)` превращает нулевую базу сравнения в `NULL`.

### Задача 1. Таблицы клиентов и заявок (2 балла)

**А.** Создайте таблицу `clients`.

| Столбец | Тип и ограничения |
|---|---|
| `client_id` | `integer`, генерируемый идентификатор, первичный ключ |
| `client_code` | `text`, обязательный уникальный код |
| `full_name` | `text`, обязательное имя |

**Б.** Создайте таблицу `loan_applications`.

| Столбец | Тип и ограничения |
|---|---|
| `application_id` | `integer`, генерируемый идентификатор, первичный ключ |
| `client_id` | `integer`, обязательная ссылка на `clients(client_id)` |
| `amount` | `numeric(12, 2)`, обязательная положительная сумма |
| `status` | `text`, обязательный статус, по умолчанию `'pending'` |
| `created_on` | `date`, обязательная дата подачи |

При обновлении ключа клиента обновляйте ссылку. Запретите удаление клиента,
пока у него есть заявки.

**А. Таблица клиентов**

In [ ]:
%%sql
-- Напишите решение

**Б. Таблица заявок**

In [ ]:
%%sql
-- Напишите решение

### Задача 2. Изменение схемы таблицы (1 балл)

**А.** Добавьте в `loan_applications` столбец `risk_score` типа `numeric(5, 2)`.
Переименуйте его в `risk_points` и расширьте до `numeric(6, 2)`. Отдельными
командами задайте значение по умолчанию 0 и запретите `NULL`. Добавьте
именованное ограничение `loan_risk_range`, разрешающее значения от 0 до 100 включительно.

**Б.** Оценку риска решили хранить отдельно. Удалите `loan_risk_range` по имени,
затем удалите `risk_points`.

**А. Изменение столбца**

In [ ]:
%%sql
-- Напишите решение

**Б. Удаление ограничения и столбца**

In [ ]:
%%sql
-- Напишите решение

### Задача 3. Загрузка пробных данных (1,5 балла)

**А.** Добавьте в `clients` пять строк через один `INSERT … VALUES`.
Идентификаторы сгенерирует PostgreSQL.

| Код | Имя |
|---|---|
| C01 | Анна Лескова |
| C02 | Борис Орлов |
| C03 | Вера Полянская |
| C04 | Денис Волков |
| C05 | Елена Лукина |

**Ячейка подготовки ниже** создаёт таблицу импорта `application_import` с такими данными.

| Код клиента | Сумма | Статус при загрузке | Дата |
|---|---:|---|---|
| C01 | 10000 | по умолчанию | 2026-01-05 |
| C02 | 25000 | approved | 2026-01-12 |
| C03 | 5000 | по умолчанию | 2026-01-12 |
| C03 | 5000 | по умолчанию | 2026-01-12 |
| C04 | 30000 | approved | 2026-01-26 |
| C05 | 15000 | по умолчанию | 2026-02-02 |
| C01 | 12000 | по умолчанию | 2026-02-09 |

**Б.** После подготовки перенесите в `loan_applications` заявки без явного статуса.
Используйте `INSERT … SELECT`. Получите `client_id` по коду клиента и дайте
столбцу `status` получить значение по умолчанию.

**В.** Перенесите заявки с заданным статусом через `INSERT … SELECT`,
передав `status` из источника. Дубликат пока сохраните.

**А. Клиенты**

In [ ]:
%%sql
-- Напишите решение

**Подготовка данных для пунктов Б и В.** Выполните эту ячейку.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS application_import (
    client_code text NOT NULL,
    amount numeric(12, 2) NOT NULL,
    status text,
    created_on date NOT NULL
);
TRUNCATE TABLE application_import;
INSERT INTO application_import (client_code, amount, status, created_on)
VALUES
    ('C01', 10000, NULL,       DATE '2026-01-05'),
    ('C02', 25000, 'approved', DATE '2026-01-12'),
    ('C03',  5000, NULL,       DATE '2026-01-12'),
    ('C03',  5000, NULL,       DATE '2026-01-12'),
    ('C04', 30000, 'approved', DATE '2026-01-26'),
    ('C05', 15000, NULL,       DATE '2026-02-02'),
    ('C01', 12000, NULL,       DATE '2026-02-09');

**Б. Заявки со статусом по умолчанию**

In [ ]:
%%sql
-- Напишите решение

**В. Заявки с заданным статусом**

In [ ]:
%%sql
-- Напишите решение

### Задача 4. Изменение статуса заявки (1 балл)

Заявка клиента `C01` от **5 января 2026 года** отклонена. Обновите только эту
заявку, установив статус `'rejected'`. Получите идентификатор клиента по коду.
У того же клиента есть другая заявка, её статус должен сохраниться.

**Изменение статуса**

In [ ]:
%%sql
-- Напишите решение

### Задача 5. Удаление дублей и очистка импорта (1 балл)

**А.** Заявки считаются дублями, если совпадают клиент, сумма, статус и дата.
В каждой такой группе оставьте заявку с наименьшим `application_id`.
Используйте `DELETE` и подзапрос, в котором строки внутри группы
нумеруются через `ROW_NUMBER()`.

**Б.** Очистите `application_import` командой `TRUNCATE`.

**В.** Удалите таблицу импорта командой `DROP TABLE IF EXISTS`.
Объясните, чем результат `TRUNCATE` отличается от `DROP TABLE`.

Если бы таблицу `clients` удалили через `DROP TABLE … CASCADE`, что произошло бы
с внешним ключом и строками `loan_applications`? Объясните без запуска этой команды.

**А. Удаление дублей**

In [ ]:
%%sql
-- Напишите решение

**Б. Очистка таблицы импорта**

In [ ]:
%%sql
-- Напишите решение

**В. Удаление таблицы импорта**

In [ ]:
%%sql
-- Напишите решение

**Ваш вывод**

### Задача 6. Обычное и материализованное представления (1,5 балла)

**А.** Создайте обычное представление `client_loans` со сводкой по всем заявкам.
Для каждого клиента с заявками покажите `client_id`, `client_code`, `full_name`,
`applications_count` и `avg_amount` (среднюю сумму, округлённую до двух знаков).

**Б.** Замените определение так, чтобы сводка считалась только по статусу
`'pending'`. Сохраните выходные столбцы и их типы.

**В.** Создайте материализованное представление `client_loans_snapshot`
с тем же результатом. Запишите в его определении тот же запрос к базовым таблицам,
что и в пункте Б. Сравните планы чтения двух объектов при фильтре
`avg_amount >= 5000` через `EXPLAIN (ANALYZE, COSTS OFF)`.

**Г.** Запустите готовую ячейку после планов: она меняет статус заявки
`C05` от **2 февраля 2026 года** на `'approved'`. Прочитайте оба представления
для `C05` и объясните разницу. Обновите снимок и снова запросите эту строку
из него. Для каждого чтения и для обновления предусмотрена своя ячейка.

**Д.** Удалите обычное представление `client_loans`. Какие данные сохранятся?

**А. Обычное представление**

In [ ]:
%%sql
-- Напишите решение

**Б. Замена запроса**

In [ ]:
%%sql
-- Напишите решение

**В. Материализованное представление**

In [ ]:
%%sql
-- Напишите решение

**В. План чтения обычного представления**

In [ ]:
%%sql
-- Напишите решение

**В. План чтения снимка**

In [ ]:
%%sql
-- Напишите решение

**Изменение данных для пункта Г.** Выполните эту ячейку.

In [ ]:
%%sql
DO $$
BEGIN
    IF to_regclass('loan_applications') IS NOT NULL THEN
        UPDATE loan_applications AS a
        SET status = 'approved'
        FROM clients AS c
        WHERE a.client_id = c.client_id
          AND c.client_code = 'C05'
          AND a.created_on = DATE '2026-02-02';
    END IF;
END
$$;

**Г. Чтение обычного представления**

In [ ]:
%%sql
-- Напишите решение

**Г. Чтение снимка до обновления**

In [ ]:
%%sql
-- Напишите решение

**Г. Обновление снимка**

In [ ]:
%%sql
-- Напишите решение

**Г. Чтение снимка после обновления**

In [ ]:
%%sql
-- Напишите решение

**Д. Удаление обычного представления**

In [ ]:
%%sql
-- Напишите решение

**Ваш вывод**

### Задача 7. Недельная динамика суммы заявок (2 балла)

**А.** Создайте материализованное представление `weekly_loan_growth`.
Постройте календарь с понедельника **5 января** по понедельник **9 февраля 2026 года**
включительно. Для каждой недели посчитайте сумму заявок всех статусов.
Учитывайте дату подачи, понедельник входит в неделю. Если заявок не было, сумма равна 0.

Выходные столбцы:

| Столбец | Значение |
|---|---|
| `week_start` | Понедельник недели, тип `date` |
| `weekly_total` | Сумма заявок за неделю |
| `weekly_growth_pct` | Прирост к предыдущей неделе в процентах |
| `four_week_growth_pct` | Прирост к неделе на четыре недели раньше, в процентах |

Для обоих процентов используйте `100 × (текущая сумма − базовая сумма) / базовая сумма`.
Округляйте итог до двух знаков. Если базовой недели нет или её сумма равна нулю,
выводите `NULL`.

**Б.** Прочитайте представление, отсортировав строки по `week_start` по возрастанию.

**А. Материализованное представление**

In [ ]:
%%sql
-- Напишите решение

**Б. Чтение результата**

In [ ]:
%%sql
-- Напишите решение

## Необязательная очистка

После занятия здесь можно удалить учебную схему со всеми объектами этой сессии.
Для этого поменяйте `CLEAN_UP` на `True` и выполните ячейку. Для повторного
решения верните `False` и запустите ноутбук сверху.

In [ ]:
CLEAN_UP = False
if CLEAN_UP and "practice_schema" in globals():
    get_ipython().run_cell_magic("sql", "", f"DROP SCHEMA {practice_schema} CASCADE;")
    del practice_schema